In [0]:
%run ./01_setup_volumen

In [0]:
import polars as pl

In [0]:
orders_path = str(settings.bronze / "orders" / f"run_date={settings.run_date}" / "*.parquet")
customers_path = str(settings.bronze / "customers" / f"run_date={settings.run_date}" / "*.parquet")
products_path = str(settings.bronze / "products" / f"run_date={settings.run_date}" / "*.parquet")
payments_path = str(settings.bronze / "payments" / f"run_date={settings.run_date}" / "*.parquet")

print(orders_path)
print(customers_path)
print(products_path)
print(payments_path)


In [0]:
orders_lf = (
    pl.scan_parquet(orders_path)
    .select(["order_id", "customer_id", "product_id", "quantity", "order_ts", "status"])
    .with_columns([
        pl.col("order_id").cast(pl.Int64),
        pl.col("customer_id").cast(pl.Int64),
        pl.col("product_id").cast(pl.Int64),
        pl.col("quantity").cast(pl.Int64),
        pl.col("order_ts"),
        pl.col("status").cast(pl.Utf8),
    ])
    .filter(pl.col("quantity") > 0)
)


In [0]:
orders_lf

In [0]:
df_orders = orders_lf.collect()
df_orders.head()

In [0]:
customers_lf = (
    pl.scan_parquet(customers_path)
    .select(["customer_id", "country", "signup_date"])
    .with_columns([
        pl.col("customer_id").cast(pl.Int64),
        pl.col("country").cast(pl.Utf8),
        pl.col("signup_date"),
    ])
)

In [0]:

products_lf = (
    pl.scan_parquet(products_path)
    .select(["product_id", "category", "unit_price", "active"])
    .with_columns([
        pl.col("product_id").cast(pl.Int64),
        pl.col("category").cast(pl.Utf8),
        pl.col("unit_price").cast(pl.Float64),
        pl.col("active").cast(pl.Boolean),
    ])
    .filter(pl.col("active") == True)
)

In [0]:
payments_lf = (
    pl.scan_parquet(payments_path)
    .select(["payment_id", "order_id", "amount", "currency", "payment_method", "paid_at"])
    .with_columns([
        pl.col("payment_id").cast(pl.Int64),
        pl.col("order_id").cast(pl.Int64),
        pl.col("amount").cast(pl.Float64),
        pl.col("currency").cast(pl.Utf8),
        pl.col("payment_method").cast(pl.Utf8),
        pl.col("paid_at")
    ])
)

In [0]:
silver_lf = (
    orders_lf
    .join(customers_lf, on="customer_id", how="left")
    .join(products_lf, on="product_id", how="left")
    .join(payments_lf, on="order_id", how="left")
    .with_columns([
        (pl.col("quantity") * pl.col("unit_price")).round(2).alias("gross_amount"),
        pl.col("order_ts").dt.date().alias("order_date"),
        pl.when(
            pl.col("signup_date").is_not_null() &
            ((pl.col("order_ts").dt.date() - pl.col("signup_date")).dt.total_days() <= 30)
        )
        .then(pl.lit("new"))
        .otherwise(pl.lit("existing"))
        .alias("customer_segment"),
    ])
    .select([
        "order_id",
        "order_date",
        "order_ts",
        "customer_id",
        "product_id",
        "country",
        "category",
        "customer_segment",
        "quantity",
        "unit_price",
        "gross_amount",
        "status",
        "payment_id",
        "amount",
        "currency",
        "payment_method",
        "paid_at",
    ])
)

silver_df = silver_lf.collect()

silver_out_dir = settings.silver / "orders" / f"run_date={settings.run_date}"
silver_out_dir.mkdir(parents=True, exist_ok=True)
silver_out = silver_out_dir / "part-000.parquet"
silver_df.write_parquet(silver_out)

print("Silver escrita en:", silver_out)

In [0]:
silver_df.head()

In [0]:
silver_df.head().to_pandas()